# 07 – Kết luận (Conclusion)
Tổng kết phương án tốt nhất, hạn chế, khuyến nghị ứng dụng và hướng cải thiện **dựa trên bằng chứng ở các notebook trước**.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

In [ ]:
from spam_filter.experiments import EXPERIMENTS
from spam_filter.runner import run_specs
results, pipes = run_specs(EXPERIMENTS, X_train, y_train, X_test, y_test, cfg.cv_folds)
top = results.sort_values("cv_f1_train", ascending=False).iloc[0]
print(f"Mô hình tốt nhất (theo CV trên train): {top['id']} – {top['description']}")
print(f"Test: precision={top.precision:.3f} recall={top.recall:.3f} f1={top.f1:.3f} | FP={top.FP} FN={top.FN}")
print("Baseline E0 F1 =", round(results[results.id == 'E0'].f1.iloc[0], 3))

## Minh họa khuyến nghị: đổi ngưỡng quyết định để ưu tiên Precision hoặc Recall
Với bộ lọc tin nhắn, chặn nhầm tin ham (FP) thường tệ hơn bỏ lọt spam (FN) → có thể tăng ngưỡng để Precision cao hơn.

In [ ]:
from sklearn.metrics import precision_score, recall_score
proba = pipes[top["id"]].predict_proba(X_test)[:, 1]
pd.DataFrame([{"ngưỡng": t, "precision": precision_score(y_test, proba >= t, zero_division=0),
               "recall": recall_score(y_test, proba >= t), "FP": int(((proba >= t) & (y_test == 0)).sum()),
               "FN": int(((proba < t) & (y_test == 1)).sum())} for t in (0.3, 0.5, 0.7, 0.9, 0.95)]).round(3).set_index("ngưỡng")

## 1. Kết quả chính (nhóm điền)
- Phương án tốt nhất và vì sao (nêu số liệu: CV F1, test F1, so với baseline).
- Yếu tố nào đóng góp nhiều nhất: tiền xử lý / vector hóa / thuật toán? (dẫn notebook 02, 03, 04)

## 2. Hạn chế
- Dữ liệu nhỏ, cũ (~2011), chỉ tiếng Anh; test chỉ ~128 spam → chênh lệch nhỏ chưa chắc có ý nghĩa.
- Mô hình đếm từ, không hiểu ngữ nghĩa/ngữ cảnh.
- (bổ sung)

## 3. Khuyến nghị ứng dụng
- Chọn ngưỡng theo chi phí FP/FN (bảng trên); cho người dùng báo spam để cập nhật mô hình định kỳ.
- (bổ sung)

## 4. Hướng cải thiện (dựa trên lỗi ở notebook 06)
- Đặc trưng thêm (độ dài, số chữ số), char n-gram, class weight, embeddings, dữ liệu mới.
- (bổ sung)